In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import ScalarFormatter

In [2]:
dataSet_to_name = {1:"Barreau:1983ht", 2:"O'Connell:198", 3:"Sealock:1989nx", 4:"Baran:1988tw", 5:"Bagdasaryan:1988hp", 6:"Dai - HallA:2019da", 
                   7:"Arrington:1995hs", 8:"Day:1993md", 9:"Arrington:1998psnoCC", 10:"Gaskell:2008", 11:"Whitney:1974hr", 12:"AlsamiJan05", 
                   13:"VaheJun07", 14:"Gomez74", 15:"Fomin", 16:"Yamaguchi73", 17:"Ryan84", 18:"Cyzyk:1963zz", 
                   19:"Bounin63", 20:"Photo-Daphne", 21:"Antony-Spies:1970jjs", 22:"Goldemberg64", 23:"DeForrest65", 24:"Mihovilovic:2024ymj", 25:"CLAS-e4nu", 26:"GarinoBates:1992"}
# # New Version
# dataSet_to_normalization = {1: 0.95971, 2: 0.96416, 3: 1.0744, 4: 0.99482, 5: 0.93381, 6: 1.0126, 
#                             7: 0.96716, 8: 1.0238, 9: 0.97904, 10: 0.99064, 11: 0.98384, 12: 1.0000, 
#                             13: 1.0163, 14: 1.0300, 15: 1.0190, 16: 0.95853, 17: 1.0174, 18: 1.0168, 
#                             19: 1.0794, 20: 1.0000, 21: 0.9500, 22: 1.1095, 23: 0.9310, 24: 1.0019, 
#                             25: 0.8500, 26: 1.0000, 33: 0.9980, 34: 0.9677, 35: 0.9561}
# dataSet_to_normError = {1: 0.62926E-02, 2: 0.12908E-01, 3: 0.80983E-02, 4: 0.69809E-02, 5: 0.16758E-01, 6: 0.92261E-02, 
#                         7: 0.15546E-01, 8: 0.65203E-02, 9: 0.55606E-02, 10: 0.75245E-02, 11: 0.25318E-01, 12: 0.0, 
#                         13: 0.17632E-02, 14: 0.91993E-02, 15: 0.63181E-02, 16: 0.25582E-01, 17: 0.42184E-01, 18: 0.68067E-01, 
#                         19: 0.35847E-01, 20: 0.0, 21: 0.25, 22: 0.1, 23: 0.1, 24: 0.184E-01, 
#                         25: 0.02, 26: 0.02, 33: 0.415E-01, 34: 0.173E-01, 35: 0.231E-01}
dataSet_to_normalization = {1: 0.9919, 2: 0.9787, 3: 1.06, 4: 0.9924, 5: 0.9878,
                            6: 1.0108, 7: 0.9743, 8: 1.0071, 9: 0.9888, 10: 0.9934,
                            11: 1.0149, 12: 0.9981, 13: 1.0029, 14: 1.0125, 15: 1.0046,
                            16: 1.0019, 17: 1.10, 18: 1.000, 19: 1.150, 20: 1.0,
                            21: 0.95, 22: 1.100, 23: 0.9, 24: 1.03, 25: 0.85, 26: 1.0000}
dataSet_to_normError = {1: 0.0024, 2: 0.0086, 3: 0.1000, 4: 0.0046, 5: 0.0083,
                        6: 0.0053, 7: 0.0133, 8: 0.0033, 9: 0.0034, 10: 0.0051,
                        11: 0.0153, 12: 0.0067, 13: 0.0070, 14: 0.0149, 15: 0.0031,
                        16: 0.0029, 17: 0.0130, 18: 0.2000, 19: 0.2300, 20: 0.0,
                        21: 0.25, 22: 0.1000, 23: 0.1000, 24: 0.02, 25: 0.02, 26: 0.02}
data = 'Data/C12.csv'
data_fit = 'Data/C12_Fit.csv'
data_GENIE = 'Data/C12_Genie.csv'
data_SuSAV2 = 'Data/C12_SuSAV2.csv'
pdf_file = 'C12_Cross.pdf'
elem = 'C12'
ex_cut_lower = 0
ex_cut_upper = 1000
A = 12
mass_nucleon = 0.938273
mass_nucleus = A * 0.931494
nu_axis = True
Q2_cut = 0.025

In [3]:
df = pd.read_csv(data)
df["normalization"] = df["dataSet"].map(dataSet_to_normalization)
df["normError"] = df["dataSet"].map(dataSet_to_normError)
df['system_err'] = 0.0
df['normCross'] = df['cross'] * df['normalization']
df['error'] = np.sqrt(df['error']**2 + ((df['system_err'] * df['cross'])**2))
df['normCrossError'] = df['normCross'] * np.sqrt((df['error'] / df['cross'])**2 + (df['normError'] / df['normalization'])**2)
df_fit = pd.read_csv(data_fit)
df_GENIE = pd.read_csv(data_GENIE)
df_SuSAV2 = pd.read_csv(data_SuSAV2)
value_pairs = sorted(set((row["E0"], row["ThetaDeg"], row["dataSet"]) for _, row in df.iterrows()), key=lambda x: (x[2], x[0], x[1]))
if not nu_axis:
    def cal_w2(df):
        df["ThetaRad"] = df["ThetaDeg"] * np.pi / 180
        df["sin2(T/2)"] = (np.sin(df["ThetaRad"] / 2))**2
        df["Q2"] = 4 * df["E0"] * (df["E0"] - df["nu"]) * df["sin2(T/2)"]
        df["W2original"] = mass_nucleon**2 + 2 * mass_nucleon * df["nu"] - df["Q2"]
    cal_w2(df)
    cal_w2(df_fit)
    cal_w2(df_GENIE)
    cal_w2(df_SuSAV2)
    if Q2_cut != 0:
        df_GENIE = df_GENIE[df_GENIE['Q2'] > 0.025]

In [4]:
with PdfPages(pdf_file) as pdf:
    for i in range(len(value_pairs) // 12 + 1):
        fig, axs = plt.subplots(ncols = 3, nrows = 4, figsize = (18, 18), dpi = 600) 
        for j, ax in enumerate(axs.flat):
            if i * 12 + j >= len(value_pairs):
                ax.axis('off')
                continue
            E0, ThetaDeg, dataSet = value_pairs[i * 12 + j]
            filtered_data = df[(df['E0'] == E0) & (df['ThetaDeg'] == ThetaDeg) & (df['dataSet'] == dataSet)]
            filtered_data = filtered_data.sort_values(by = 'nu')
            x = filtered_data['nu'] if nu_axis else filtered_data['W2original']
            y = filtered_data['normCross']
            yerr = filtered_data['normCrossError']
            dataSetName = dataSet_to_name[dataSet]
            normalization = dataSet_to_normalization[dataSet]
            filtered_data_fit = df_fit[(df_fit['E0'] == E0) & (df_fit['ThetaDeg'] == ThetaDeg)]
            filtered_data_fit = filtered_data_fit.sort_values(by = 'nu')
            x_fit = filtered_data_fit['nu'] if nu_axis else filtered_data_fit['W2original']
            y_fit = filtered_data_fit['sigtot']
            filtered_data_GENIE = df_GENIE[(df_GENIE['E0'] == E0) & (df_GENIE['ThetaDeg'] == ThetaDeg)]
            filtered_data_GENIE = filtered_data_GENIE.sort_values(by = 'nu')
            x_GENIE = filtered_data_GENIE['nu'] if nu_axis else filtered_data_GENIE['W2original']
            y_GENIE = filtered_data_GENIE['cross']
            filtered_data_SuSAV2 = df_SuSAV2[(df_SuSAV2['E0'] == E0) & (df_SuSAV2['ThetaDeg'] == ThetaDeg)]
            filtered_data_SuSAV2 = filtered_data_SuSAV2.sort_values(by = 'nu')
            x_SuSAV2 = filtered_data_SuSAV2['nu'] if nu_axis else filtered_data_SuSAV2['W2original']
            y_SuSAV2 = filtered_data_SuSAV2['cross']
            ax.errorbar(x, y, yerr=yerr, fmt='.', label='normCross', color='blue', markersize=8, capsize=0, alpha=1.0, zorder=1)
            ax.plot(x_fit, y_fit, label='Christy-Bodek Fit', color='red', linestyle='solid', linewidth=2, alpha=0.5, zorder=2)
            ax.scatter(x_GENIE, y_GENIE, label='GENIE', color='saddlebrown', marker='D', s=12, linewidth=0, alpha=1.0, zorder=3)
            ax.scatter(x_SuSAV2, y_SuSAV2, label='SuSAV2', color='lawngreen', marker='s', s=18, edgecolor = 'black', linewidth=0.2, alpha=1.0, zorder=4)
            if nu_axis:
                ax.set_xlabel('$\\nu \ (GeV)$')
            else:
                ax.set_xlabel('$W^2 (GeV^2)$')
            ax.set_ylabel('$\\frac{d^2 \sigma}{d\Omega d\\nu} (nb/sr/GeV)$')
            ax.set_ylim(0, None)
            ax.set_title(f'{int(dataSet)}:{dataSetName} {E0}$GeV$ {ThetaDeg}° (X {normalization:.4f})')
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_scientific(True)
            formatter.set_powerlimits((0,0))
            ax.yaxis.set_major_formatter(formatter)
            if j == 0:
                ax.legend()
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

<>:33: SyntaxWarning: invalid escape sequence '\ '
<>:36: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\ '
<>:36: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Rhys\AppData\Local\Temp\ipykernel_2412\1932549842.py:33: SyntaxWarning: invalid escape sequence '\ '
  ax.set_xlabel('$\\nu \ (GeV)$')
C:\Users\Rhys\AppData\Local\Temp\ipykernel_2412\1932549842.py:36: SyntaxWarning: invalid escape sequence '\s'
  ax.set_ylabel('$\\frac{d^2 \sigma}{d\Omega d\\nu} (nb/sr/GeV)$')
